# 08 · Link Prediction

## What this notebook covers
Link prediction asks: given the current graph, which edges are likely to exist but are unobserved? This is fundamental for:
- **Recommender systems**: "users you might know", product recommendations
- **Drug-target interaction prediction**: which molecules bind to which proteins?
- **Knowledge graph completion**: inferring missing triples

## Approaches
| Method | Description | Use when |
|--------|------------|---------|
| **Heuristic** | Common neighbours, Jaccard, Adamic-Adar | Fast baseline; works when transitivity holds |
| **Matrix factorisation** | SVD of adjacency; node2vec | No node features available |
| **GNN-based** | Encode node pairs; decode with dot product or MLP | Node features available; best accuracy |

The standard GNN link prediction pipeline:
1. Train node embeddings with a GNN (no link labels needed — use node features / structure)
2. Score edges as dot product or learned MLP of node embedding pairs
3. Train with binary cross-entropy on positive (real) and negative (sampled) edges

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import networkx as nx
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

try:
    from torch_geometric.datasets import Planetoid
    from torch_geometric.nn import GCNConv
    from torch_geometric.utils import negative_sampling, train_test_split_edges, to_undirected
    HAS_PYG = True
except ImportError:
    HAS_PYG = False

In [ ]:
if HAS_PYG:
    dataset = Planetoid(root="/tmp/Cora", name="Cora")
    data = dataset[0]
    data = train_test_split_edges(data)

    class LinkGCN(torch.nn.Module):
        def __init__(self, in_c, hid_c, out_c):
            super().__init__()
            self.conv1 = GCNConv(in_c, hid_c)
            self.conv2 = GCNConv(hid_c, out_c)

        def encode(self, x, edge_index):
            x = F.relu(self.conv1(x, edge_index))
            return self.conv2(x, edge_index)

        def decode(self, z, edge_index):
            return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

        def decode_all(self, z):
            return torch.sigmoid((z @ z.t()))

    model = LinkGCN(dataset.num_features, 64, 32)
    opt   = torch.optim.Adam(model.parameters(), lr=0.01)

    def train():
        model.train()
        opt.zero_grad()
        z = model.encode(data.x, data.train_pos_edge_index)
        pos_score = model.decode(z, data.train_pos_edge_index)
        neg_idx = negative_sampling(data.train_pos_edge_index, num_nodes=data.num_nodes,
                                     num_neg_samples=data.train_pos_edge_index.size(1))
        neg_score = model.decode(z, neg_idx)
        scores = torch.cat([pos_score, neg_score])
        labels = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))])
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward(); opt.step()
        return loss.item()

    @torch.no_grad()
    def test():
        model.eval()
        z = model.encode(data.x, data.train_pos_edge_index)
        pos_score = torch.sigmoid(model.decode(z, data.test_pos_edge_index)).cpu()
        neg_score = torch.sigmoid(model.decode(z, data.test_neg_edge_index)).cpu()
        y_true = torch.cat([torch.ones(pos_score.size(0)), torch.zeros(neg_score.size(0))])
        y_score = torch.cat([pos_score, neg_score])
        return roc_auc_score(y_true.numpy(), y_score.numpy())

    aucs = []
    for epoch in range(100):
        loss = train()
        auc  = test()
        aucs.append(auc)
    print(f"Link prediction on Cora: AUC = {aucs[-1]:.4f}")

    plt.figure(figsize=(8, 4))
    plt.plot(aucs, color="steelblue")
    plt.title(f"Link prediction AUC on Cora (final={aucs[-1]:.3f})")
    plt.xlabel("Epoch"); plt.ylabel("AUC"); plt.tight_layout()
    plt.savefig(f"{base}/08_link_prediction.png", dpi=120)
    plt.show()
else:
    # Heuristic baseline on karate club
    G = nx.karate_club_graph()
    edges = list(G.edges())
    # Remove 20% edges as test set
    np.random.shuffle(edges)
    n_test = len(edges) // 5
    test_pos = edges[:n_test]; train_edges = edges[n_test:]
    G_train = nx.Graph(); G_train.add_nodes_from(G.nodes()); G_train.add_edges_from(train_edges)

    preds = dict(nx.adamic_adar_index(G_train, test_pos))
    scores = [preds.get(e, 0) for e in test_pos]
    # Negative edges
    non_edges = list(nx.non_edges(G_train))
    np.random.shuffle(non_edges)
    neg_test = non_edges[:n_test]
    neg_scores = [0.0] * n_test

    y_true  = [1]*n_test + [0]*n_test
    y_score = scores + neg_scores
    auc = roc_auc_score(y_true, y_score)
    print(f"Adamic-Adar heuristic on Karate Club: AUC = {auc:.4f}")